# Accident detection

## Libraries & Dataset

In [ ]:
!pip install -qU roboflow ultralytics wandb

In [ ]:
from kaggle_secrets import UserSecretsClient
import wandb

user_secrets = UserSecretsClient()
roboflow_api_key = user_secrets.get_secret("RoboFlow")
wandb_api_key = user_secrets.get_secret("WandB_SafeSpace")

wandb.login(key=wandb_api_key)

In [ ]:
from roboflow import Roboflow
rf = Roboflow(api_key=roboflow_api_key)
project = rf.workspace("dataset-tfm18").project("zihan-z36um")
version = project.version(5)
dataset = version.download("yolov8")

In [ ]:
# ── W&B: Initialize run with full hyperparameter config ──────────────
EPOCHS = 12
IMGSZ  = 640
BATCH  = 16
MODEL  = "yolov8n.pt"
PROJECT = "Accident_Severity_Detection"
RUN_NAME = "v1_Accident_detection_AG"     # edit AG to your short name & start with v1 & edit the version number as you go !!!!

run = wandb.init(
    project=PROJECT,
    name=RUN_NAME,
    job_type="training",
    config = {
        "model":        MODEL,
        "pretrained":   True,
        # Training
        "epochs":          EPOCHS,
        "imgsz":           IMGSZ,
        "batch":           BATCH,
        "fraction":        1.0,     # Here we used 100% of the data
        "optimizer":       "auto",
        "lr0":             0.01,
        "lrf":             0.01,
        "momentum":        0.937,
        "weight_decay":    0.0005,
        "warmup_epochs":   3.0,
        "cos_lr":          True,
        "patience":        10,
        "box":             9.0,
        # Augmentation — stripped to avoid double-augmenting Roboflow's offline transforms
        "fliplr":          0.0,    # Roboflow already flipped
        "hsv_h":           0.0,    # Roboflow already shifted hue
        "hsv_s":           0.0,    # Roboflow already shifted saturation
        "hsv_v":           0.0,    # Roboflow already shifted brightness/exposure
        "erasing":         0.0,    # Roboflow already applied cutout
        "mosaic":          1.0,    # ✅ Keep — not in Roboflow pipeline
        "translate":       0.1,    # ✅ Keep — adds useful position variance
        "scale":           0.1,    # ✅ Keep — different from Roboflow's fixed crop
        # Dataset
        "dataset":         "zihan-z36um",
        "dataset_version": 5,
        "dataset_link":    "https://universe.roboflow.com/dataset-tfm18/zihan-z36um/dataset/5",
        "num_classes":     2,
}
)
print(f"W&B run started: {run.url}")


## Modeling

In [ ]:
from ultralytics import YOLO

cfg = wandb.config  # use values logged to W&B

model = YOLO(cfg.model)

results = model.train(
    data='/kaggle/working/License-Plate-Recognition-13/data.yaml',
    epochs=cfg.epochs,
    imgsz=cfg.imgsz,
    batch=cfg.batch,
    fraction=cfg.fraction,
    cos_lr=cfg.cos_lr,
    patience=cfg.patience,
    box=cfg.box,
    erasing=cfg.erasing,
    fliplr=cfg.fliplr,
    hsv_h=cfg.hsv_h,
    hsv_s=cfg.hsv_s,
    hsv_v=cfg.hsv_v,
    scale=cfg.scale,
    project=PROJECT,
    name=RUN_NAME,
    plots=True,
)

In [ ]:
# ── W&B: Log final validation metrics ─────────────────────────────────
metrics_dict = results.results_dict

final_metrics = {
    "final/precision":    metrics_dict.get("metrics/precision(B)", 0),
    "final/recall":       metrics_dict.get("metrics/recall(B)",    0),
    "final/mAP50":        metrics_dict.get("metrics/mAP50(B)",     0),
    "final/mAP50-95":     metrics_dict.get("metrics/mAP50-95(B)",  0),
    "final/box_loss":     metrics_dict.get("val/box_loss",         0),
    "final/cls_loss":     metrics_dict.get("val/cls_loss",         0),
    "final/dfl_loss":     metrics_dict.get("val/dfl_loss",         0),
    "final/fitness":      results.fitness,
}
wandb.log(final_metrics)

# Also write them as W&B summary so they show in the runs table
for k, v in final_metrics.items():
    wandb.run.summary[k] = v

print("Logged metrics:")
for k, v in final_metrics.items():
    print(f"  {k}: {v:.4f}")


In [ ]:
# ── W&B: Log training plots & validation images ───────────────────────
from pathlib import Path

save_dir = Path(results.save_dir)

plot_files = {
    "confusion_matrix":           save_dir / "confusion_matrix.png",
    "confusion_matrix_normalized": save_dir / "confusion_matrix_normalized.png",
    "BoxPR_curve":                   save_dir / "BoxPR_curve.png",
    "BoxF1_curve":                   save_dir / "BoxF1_curve.png",
    "BoxP_curve":                    save_dir / "BoxP_curve.png",
    "BoxR_curve":                    save_dir / "BoxR_curve.png",
    "results":                    save_dir / "results.png",
    "labels":                     save_dir / "labels.jpg"
}

wandb_images = {}
for name, path in plot_files.items():
    if path.exists():
        wandb_images[f"plots/{name}"] = wandb.Image(str(path), caption=name)
        print(f"  ✓ {name}")
    else:
        print(f"  ✗ {name} not found")

# Validation batch predictions (ground-truth vs predictions)
for img_path in sorted(save_dir.glob("val_batch*.jpg")):
    wandb_images[f"val_batches/{img_path.stem}"] = wandb.Image(
        str(img_path), caption=img_path.stem
    )
    print(f"  ✓ {img_path.stem}")

wandb.log(wandb_images)
print(f"\nLogged {len(wandb_images)} images/plots to W&B.")


In [ ]:
# ── W&B: Save best model as a versioned artifact ──────────────────────
best_pt = save_dir / "weights" / "best.pt"
last_pt = save_dir / "weights" / "last.pt"

artifact = wandb.Artifact(
    name="license_plate_detector",
    type="model",
    description="YOLOv26l fine-tuned for license plate detection",
    metadata={
        "mAP50":     wandb.run.summary.get("final/mAP50"),
        "mAP50-95":  wandb.run.summary.get("final/mAP50-95"),
        "precision": wandb.run.summary.get("final/precision"),
        "recall":    wandb.run.summary.get("final/recall"),
        "epochs":    cfg.epochs,
        "imgsz":     cfg.imgsz,
        "dataset_link": cfg.dataset_link,
    }
)

if best_pt.exists():
    artifact.add_file(str(best_pt), name="best.pt")
if last_pt.exists():
    artifact.add_file(str(last_pt), name="last.pt")

wandb.log_artifact(artifact)
artifact.wait()  # ← block until artifact is fully logged on the W&B server
print(f"Model artifact logged: {artifact.name}:{artifact.version}")


In [ ]:
# Finish the WandB run
wandb.finish()